# Imports necesarios

In [2]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Cargar CSVs

In [3]:
nodes = pd.read_csv("../data/nodes.csv")
edges = pd.read_csv("../data/edges.csv")

print("Dimensión de nodes:", nodes.shape)
print("Dimensión de edges:", edges.shape)

display(nodes.head())
display(edges.head())

Dimensión de nodes: (156422, 6)
Dimensión de edges: (300386, 2)


,spotify_id,name,followers,popularity,genres,chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,"['nordic house', 'russelater']",['no (3)']
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,"['christlicher rap', 'german hip hop']","['at (44)', 'de (111)', 'lu (22)', 'ch (31)', ..."
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,[],['de (1)']
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,"['dancehall', 'lovers rock', 'modern reggae', ...","['at (1)', 'de (1)']"
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,"['classic swedish pop', 'norrbotten indie', 's...",['se (6)']


,id_0,id_1
0,76M2Ekj8bG8W7X2nbx2CpF,7sfl4Xt5KmfyDs2T3SVSMK
1,0hk4xVujcyOr6USD95wcWb,7Do8se3ZoaVqUt3woqqSrD
2,38jpuy3yt3QIxQ8Fn1HTeJ,4csQIMQm6vI2A2SCVDuM2z
3,6PvcxssrQ0QaJVaBWHD07l,6UCQYrcJ6wab6gnQ89OJFh
4,2R1QrQqWuw3IjoP5dXRFjt,4mk1ScvOUkuQzzCZpT6bc0


# Crear has_chart_hits

In [4]:
nodes["has_chart_hits"] = nodes["chart_hits"].notna().astype(int)

print(nodes["has_chart_hits"].value_counts())
print(nodes["has_chart_hits"].value_counts(normalize=True))

has_chart_hits
0    136781
1     19641
Name: count, dtype: int64
has_chart_hits
0    0.874436
1    0.125564
Name: proportion, dtype: float64


In [5]:
display(nodes[["name", "chart_hits", "has_chart_hits"]].head(20))

,name,chart_hits,has_chart_hits
0,Byklubben,['no (3)'],1
1,Kontra K,"['at (44)', 'de (111)', 'lu (22)', 'ch (31)', ...",1
2,Maxim,['de (1)'],1
3,Christopher Martin,"['at (1)', 'de (1)']",1
4,Jakob Hellman,['se (6)'],1
5,Madh,['it (2)'],1
6,Juice,['se (4)'],1
7,Nehuda,['fr (1)'],1
8,VovaZiLvova,['ua (1)'],1
9,Nata Record,['do (1)'],1


In [6]:
print(nodes[["followers", "popularity"]].isna().sum())
display(nodes[["followers", "popularity"]].describe())

followers     4
popularity    0
dtype: int64


,followers,popularity
count,1.564180e+05,156422.000000
mean,8.622371e+04,21.157497
std,9.401001e+05,18.338290
min,0.000000e+00,0.000000
25%,2.400000e+01,4.000000
50%,3.630000e+02,18.000000
75%,6.258000e+03,34.000000
max,1.021569e+08,100.000000


In [7]:
nodes["followers"] = nodes["followers"].fillna(0)
nodes["popularity"] = nodes["popularity"].fillna(0)

In [8]:
print(nodes[["followers", "popularity"]].isna().sum())

followers     0
popularity    0
dtype: int64


In [9]:
G = nx.Graph()

G.add_nodes_from(nodes["spotify_id"])

G.add_edges_from(
    edges[["id_0", "id_1"]].itertuples(index=False, name=None)
)

print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())

Número de nodos: 156326
Número de aristas: 300386


In [10]:
valid_nodes = set(nodes["spotify_id"])
G = G.subgraph(valid_nodes).copy()

print("Número de nodos finales:", G.number_of_nodes())
print("Número de aristas finales:", G.number_of_edges())

Número de nodos finales: 156320
Número de aristas finales: 300379


In [11]:
print("Es dirigido:", G.is_directed())
print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())
print("Número de componentes conexas:", nx.number_connected_components(G))

Es dirigido: False
Número de nodos: 156320
Número de aristas: 300379
Número de componentes conexas: 4338


In [12]:
largest_cc = max(nx.connected_components(G), key=len)


print("Tamaño de la componente gigante:", len(largest_cc))
print("Porcentaje de nodos en componente gigante:", len(largest_cc) / G.number_of_nodes())

G = G.subgraph(largest_cc).copy()
nodes = nodes[nodes['spotify_id'].isin(G.nodes())].copy()

print("Nuevos nodos en G:", G.number_of_nodes())
print("Nuevas filas en nodes:", nodes.shape[0])

Tamaño de la componente gigante: 148380
Porcentaje de nodos en componente gigante: 0.9492067553735927
Nuevos nodos en G: 148380
Nuevas filas en nodes: 148473


In [13]:
degrees = [degree for node, degree in G.degree()]

average_degree = sum(degrees) / len(degrees)

print("Grado medio:", average_degree)
print("Grado máximo:", max(degrees))
print("Grado mínimo:", min(degrees))

Grado medio: 4.000040436716539
Grado máximo: 1781
Grado mínimo: 1


In [14]:
degree = dict(G.degree())

nodes["degree"] = nodes["spotify_id"].map(degree).fillna(0)

display(nodes[["name", "degree"]].sort_values("degree", ascending=False).head(10))

,name,degree
12406,Johann Sebastian Bach,1781
18735,Traditional,1371
5609,Mc Gw,858
13370,MC MN,632
11577,Jean Sibelius,580
2654,Armin van Buuren,513
8030,Gucci Mane,509
13017,Steve Aoki,498
19434,Snoop Dogg,495
7956,Diplo,494


In [15]:
degree_centrality = nx.degree_centrality(G)

nodes["degree_centrality"] = nodes["spotify_id"].map(degree_centrality).fillna(0)

display(
    nodes[["name", "degree", "degree_centrality"]]
    .sort_values("degree_centrality", ascending=False)
    .head(10)
)

,name,degree,degree_centrality
12406,Johann Sebastian Bach,1781,0.012003
18735,Traditional,1371,0.009240
5609,Mc Gw,858,0.005782
13370,MC MN,632,0.004259
11577,Jean Sibelius,580,0.003909
2654,Armin van Buuren,513,0.003457
8030,Gucci Mane,509,0.003430
13017,Steve Aoki,498,0.003356
19434,Snoop Dogg,495,0.003336
7956,Diplo,494,0.003329


In [16]:
clustering = nx.clustering(G)

nodes["clustering"] = nodes["spotify_id"].map(clustering).fillna(0)

display(
    nodes[["name", "degree", "clustering"]]
    .sort_values("clustering", ascending=False)
    .head(10)
)

,name,degree,clustering
30089,Franek Kimono,2,1.0
30097,El Cherry Scoom,2,1.0
30112,Fred Astaire,2,1.0
30129,Bénabar,2,1.0
29965,AstroWilk,2,1.0
30160,Olivia Penalva,2,1.0
156296,Sullee J,2,1.0
156298,Ricardo Quijano,2,1.0
156302,DJ Harry Lotay,2,1.0
156310,Sunsun,2,1.0


In [17]:
pagerank = nx.pagerank(G, alpha=0.85)

nodes["pagerank"] = nodes["spotify_id"].map(pagerank).fillna(0)

display(
    nodes[["name", "degree", "pagerank"]]
    .sort_values("pagerank", ascending=False)
    .head(10)
)

,name,degree,pagerank
12406,Johann Sebastian Bach,1781,0.003823
18735,Traditional,1371,0.002927
11577,Jean Sibelius,580,0.001164
5609,Mc Gw,858,0.001064
13370,MC MN,632,0.000837
17437,הכוכב הבא,377,0.000808
4518,John Williams,415,0.000769
9595,A.R. Rahman,463,0.000692
2654,Armin van Buuren,513,0.000677
19434,Snoop Dogg,495,0.000637


## Resumen final de las estadísticas

In [18]:
display(
    nodes[
        [
            "degree",
            "degree_centrality",
            "clustering",
            "pagerank"
        ]
    ].describe()
)

,degree,degree_centrality,clustering,pagerank
count,148473.000000,148473.000000,148473.000000,148473.000000
mean,4.008109,0.000027,0.085068,0.000007
std,14.689397,0.000099,0.242412,0.000022
min,1.000000,0.000007,0.000000,0.000002
25%,1.000000,0.000007,0.000000,0.000002
50%,1.000000,0.000007,0.000000,0.000003
75%,2.000000,0.000013,0.000000,0.000004
max,1781.000000,0.012003,1.000000,0.003823


## Calcular comunidades

In [19]:
import networkx.algorithms.community as nx_comm

print("Calculando comunidades con Louvain (esto puede tardar un par de minutos)...")

# 1. Ejecutar el algoritmo de Louvain sobre nuestro grafo limpio
# Le pasamos una semilla (seed) para que el resultado sea reproducible
communities = nx_comm.louvain_communities(G, seed=42)

# 2. Crear un diccionario para mapear cada artista a su nueva comunidad
community_dict = {}
for id_comunidad, grupo_nodos in enumerate(communities):
    for node in grupo_nodos:
        community_dict[node] = id_comunidad

# 3. Asignarlo a la columna del dataframe
nodes["community"] = nodes["spotify_id"].map(community_dict).fillna(-1).astype(int)

print("Número de comunidades reales encontradas:", nodes["community"].nunique())
print("Distribución de los artistas en las comunidades principales:")
print(nodes["community"].value_counts().head(10))

Calculando comunidades con Louvain (esto puede tardar un par de minutos)...
Número de comunidades reales encontradas: 113
Distribución de los artistas en las comunidades principales:
community
62     23732
2      12661
64      9767
75      8636
81      7125
25      6380
9       4559
112     4298
35      4251
86      3789
Name: count, dtype: int64


## Creación de features_final.csv

In [20]:
features = nodes[
    [
        "spotify_id",
        "name",
        "followers",
        "popularity",
        "degree",
        "degree_centrality",
        "clustering",
        "pagerank",
        "community",
        "has_chart_hits"
    ]
].copy()

display(features.head())
print(features.shape)

,spotify_id,name,followers,popularity,degree,degree_centrality,clustering,pagerank,community,has_chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,2,0.000013,0.000000,0.000006,35,1
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,64,0.000431,0.067956,0.000075,9,1
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,8,0.000054,0.107143,0.000013,9,1
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,39,0.000263,0.037787,0.000051,86,1
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,2,0.000013,0.000000,0.000007,35,1


(148473, 10)


In [21]:
print(features.isna().sum())

spotify_id           0
name                 4
followers            0
popularity           0
degree               0
degree_centrality    0
clustering           0
pagerank             0
community            0
has_chart_hits       0
dtype: int64


In [22]:
features.to_csv("../data/features_final.csv", index=False)

In [23]:
test = pd.read_csv("../data/features_final.csv")

display(test.head())
print(test.shape)
print(test.isna().sum())

,spotify_id,name,followers,popularity,degree,degree_centrality,clustering,pagerank,community,has_chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,2,0.000013,0.000000,0.000006,35,1
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,64,0.000431,0.067956,0.000075,9,1
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,8,0.000054,0.107143,0.000013,9,1
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,39,0.000263,0.037787,0.000051,86,1
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,2,0.000013,0.000000,0.000007,35,1


(148473, 10)
spotify_id           0
name                 4
followers            0
popularity           0
degree               0
degree_centrality    0
clustering           0
pagerank             0
community            0
has_chart_hits       0
dtype: int64


In [24]:
features["name"] = features["name"].fillna("Unknown")
features.to_csv("../data/features_final.csv", index=False)

test = pd.read_csv("../data/features_final.csv")

print(test.shape)
print(test.isna().sum())

(148473, 10)
spotify_id           0
name                 0
followers            0
popularity           0
degree               0
degree_centrality    0
clustering           0
pagerank             0
community            0
has_chart_hits       0
dtype: int64


In [25]:
print(test["has_chart_hits"].value_counts())
print(test["has_chart_hits"].value_counts(normalize=True))

has_chart_hits
0    133249
1     15224
Name: count, dtype: int64
has_chart_hits
0    0.897463
1    0.102537
Name: proportion, dtype: float64


In [26]:
display(test.describe())

,followers,popularity,degree,degree_centrality,clustering,pagerank,community,has_chart_hits
count,1.484730e+05,148473.000000,148473.000000,148473.000000,148473.000000,148473.000000,148473.000000,148473.000000
mean,8.680848e+04,21.254929,4.008109,0.000027,0.085068,0.000007,54.843480,0.102537
std,9.513165e+05,18.317882,14.689397,0.000099,0.242412,0.000022,31.164334,0.303354
min,0.000000e+00,0.000000,1.000000,0.000007,0.000000,0.000002,0.000000,0.000000
25%,2.300000e+01,4.000000,1.000000,0.000007,0.000000,0.000002,25.000000,0.000000
50%,3.510000e+02,18.000000,1.000000,0.000007,0.000000,0.000003,62.000000,0.000000
75%,6.154000e+03,34.000000,2.000000,0.000013,0.000000,0.000004,76.000000,0.000000
max,1.021569e+08,100.000000,1781.000000,0.012003,1.000000,0.003823,112.000000,1.000000
